# Notebook 5 — A real Compass run on a balanced subset of Th17 cells

**Where we are.** Notebook 4 proved that Compass runs correctly on your machine (3 cells × 100 reactions, identical to the authors' numbers). This notebook is the next step: a **real run on 10 Th17p + 10 Th17n cells** with the paper's exact settings, followed by an analysis that asks the paper's question — *which reactions differ between pathogenic (Th17p) and non-pathogenic (Th17n) cells?* — on **your own Compass output**.

### What happens, in order
| Part | What | Needs Compass? | Time |
|---|---|---|---|
| **A** | Set-up and pre-flight (the same checks as Notebook 4, levels 1–3) | – | < 1 min |
| **B** | Choose the experiment: which cells, which reactions; **cost estimate**; *how many cells are enough?* (simulation on the authors' data) | no | ≈ 10 s |
| **C** | Stage 1 on all 290 cells, then **launch stage 2 in the background** | yes | stage 1 ≈ 0.5 min, launch is instant |
| **D** | Progress report — re-run this cell whenever you like (you can close the notebook) | – | instant |
| **E** | Is the output right? Cell-by-cell comparison with the authors' `reactions.tsv` | – | seconds |
| **F** | The biology: Th17p vs Th17n on your cells, compared with all 290 cells | – | seconds |
| **G** | Save tables (Excel format `;` and `,`) and figures; timing and projection to bigger runs | – | seconds |

**Run Parts A → D now, then come back and run E → G when Part D says the run is finished.**

### Why stage 1 uses all 290 cells even though only 20 cells go through stage 2
A cell's penalties are smoothed with its **k = 10 nearest neighbours** among the cells given to Compass (λ = 0.25) **[Paper, Algorithm 2]**. Give Compass only 20 cells and the neighbours — hence the penalties — change. So the notebook lets Compass compute the penalties of **all 290** cells (stage 1, cheap), and sends only the **chosen 20** through stage 2 (the linear programs, expensive). Their penalties are then exactly those of a full run.

### A trap this notebook avoids **[Repo: main.py `save_single_sample_penalties`, algorithm.py `singleSampleCompass`]**
Compass saves the penalties as files `sample0 … sample289` **by the column position** in the data file, and stage 2 reads `sample{i}` for the *i-th column of the file it is given*. If we simply handed Compass a file with 20 columns picked from the middle of 290, cell *j* would silently receive the penalties of cell *j* of the original file — **the wrong cell**, with no error message. Part C therefore copies the right penalty files to the right positions and then **verifies by cell name** that every one of them is correct. Part E would also expose a mix-up, because the output would disagree with the authors' file.

### How every statement is sourced
**[Paper]** = the uploaded paper · **[Repo]** = a file in the Compass repository (named) · **[Inference]** = my own reading, verify before relying on it.

## Part A · Set-up
### A.1 Imports, environment check, and locating the Compass clone
**Run the two cells below first — every time the kernel restarts.** Pick your `che411` environment as the kernel (top right in VS Code). The only line you may need to edit is `REPO_PATH` in the second cell.

In [ ]:
import os, sys, re, json, time, shutil, tempfile, platform, subprocess, warnings
from pathlib import Path

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from scipy import sparse
    from scipy.optimize import linprog
except ModuleNotFoundError as e:
    _pip = {"sklearn": "scikit-learn"}.get(e.name.split(".")[0], e.name.split(".")[0])
    raise ModuleNotFoundError(
        f"The package '{e.name}' is not installed in the Python that runs this notebook:\n  {sys.executable}\n"
        f"Fix: copy this line into a terminal, then restart the kernel and run this cell again:\n"
        f"  \"{sys.executable}\" -m pip install {_pip}\n"
        "(If that path is not your che411 environment, pick che411 as the kernel instead.)") from None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 30, "display.width", 160, "display.precision", 4)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

# ---- environment check: lowest versions this notebook was written against (see requirements.txt) ----
from importlib.metadata import version, PackageNotFoundError
MIN_VERSIONS = {"numpy": "1.21", "pandas": "1.4", "scipy": "1.9", "matplotlib": "3.5"}
def _major_minor(v):
    return tuple(int(x) for x in re.findall(r"\d+", v)[:2])
print(f"Python {platform.python_version()}   |   this kernel runs: {sys.executable}")
too_old = []
for pkg, lowest in MIN_VERSIONS.items():
    try:
        have = version(pkg)
    except PackageNotFoundError:
        have = None
    ok = have is not None and _major_minor(have) >= _major_minor(lowest)
    print(f"  {'OK ' if ok else 'OLD'}  {pkg:<13} {have or 'unknown':<10} (needs >= {lowest})")
    if not ok:
        too_old.append(pkg)
if sys.version_info < (3, 10):
    print("! Python 3.10+ is recommended (the Compass repo itself requires it).")
if too_old:
    print("\n! Upgrade with:   python -m pip install -U " + " ".join(too_old) + "   then restart the kernel and re-run.")

In [ ]:
# =========================================================================================================
#  >>>  THE ONLY LINE YOU MAY NEED TO EDIT  <<<
#  The folder created by `git clone https://github.com/wagnerlab-berkeley/Compass.git`
#  (the one that contains README.md, compass/ and docs/).   macOS ignores letter case: ".../compass" also works.
REPO_PATH  = "/Users/willnath/Compass"
AUTO_CLONE = False          # True = if nothing is found, clone the repo into ./Compass (about 400 MB)
# =========================================================================================================

REPO_URL = "https://github.com/wagnerlab-berkeley/Compass.git"
MARKER   = "docs/notebooks/extdata/Th17/reactions.tsv"        # a file that exists only in the repo's top folder

def _is_repo(p):
    try:
        return (Path(p).expanduser() / MARKER).exists()
    except OSError:
        return False

def find_repo():
    """Return (repo_folder, how_it_was_found). Order: REPO_PATH, $COMPASS_REPO, this folder + parents, usual places."""
    cands = [("REPO_PATH", REPO_PATH), ("$COMPASS_REPO", os.environ.get("COMPASS_REPO"))]
    for up in [Path.cwd(), *Path.cwd().parents]:
        cands += [("the notebook's folder / a parent", up), ("a 'Compass' sub-folder", up / "Compass")]
    for sub in ["Compass", "compass", "Documents/Compass", "Documents/GitHub/Compass", "Desktop/Compass",
                "Downloads/Compass", "projects/Compass", "code/Compass", "GitHub/Compass"]:
        cands.append(("a usual place under your home folder", Path.home() / sub))
    for how, c in cands:
        if c and _is_repo(c):
            return Path(c).expanduser().resolve(), how
    return None, None

REPO, how_found = find_repo()
if REPO is None and AUTO_CLONE:
    print("Compass repo not found locally - cloning (shallow) into ./Compass ...")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "Compass"], check=True)
    REPO, how_found = find_repo()
if REPO is None:
    raise FileNotFoundError(
        f"Could not find the Compass repository (looked for the file {MARKER}).\n"
        f"  REPO_PATH is currently '{REPO_PATH}'.\n"
        "  Fix 1: set REPO_PATH (above) to your clone. In a terminal:  cd <your clone>  and run  pwd  to print the path.\n"
        f"  Fix 2: not cloned yet?  git clone {REPO_URL}   (or set AUTO_CLONE = True)")
if how_found != "REPO_PATH":
    print(f"! REPO_PATH ('{REPO_PATH}') does not contain the repo; using the clone found via: {how_found}.\n")

DATA = REPO / "docs/notebooks/extdata"                         # Th17/ and RECON2/ live here
RES       = REPO / "compass/Resources"
# ---- every file this notebook reads (path relative to the repo folder -> what it is) ----
REQUIRED = {
    'docs/notebooks/extdata/Th17/linear_gene_expression_matrix.tsv': "the paper's input: gene x 290-cell TPM matrix",
    'docs/notebooks/extdata/Th17/cell_metadata.csv': 'which cell is Th17p / Th17n',
    'docs/notebooks/extdata/Th17/reactions.tsv': "the AUTHORS' Compass output = the reference we compare with",
    'docs/notebooks/extdata/RECON2/reaction_metadata.csv': 'Recon2 reaction names, pathways (subsystems)',
    'compass/main.py': 'Compass command-line program',
    'compass/Resources/COMPASS/RECON2_mat/default-media/preprocess.json': 'cached maximal fluxes v_opt (stage 0)',
    'compass/Resources/Metabolic Models/RECON2_mat/model/model.S.json': 'Recon2 model files',
}
OPTIONAL = {}

ALL_FILES = {**REQUIRED, **OPTIONAL}
missing, w = [], max(len(k) for k in ALL_FILES)
for rel, what in ALL_FILES.items():
    p = REPO / rel
    ok = p.is_file() and p.stat().st_size > 0
    tag = "OK " if ok else ("-- " if rel in OPTIONAL else "!! ")
    size = f"{p.stat().st_size / 1e6:8.2f} MB" if ok else "     MISSING"
    print(f"{tag}{rel:<{w}} {size}   {what}")
    if not ok and rel in REQUIRED:
        missing.append(rel)
if missing:
    raise FileNotFoundError(
        f"{len(missing)} required file(s) are missing inside {REPO}:\n  " + "\n  ".join(missing) +
        "\nLikely cause: the clone is incomplete or REPO_PATH points to the wrong folder."
        f"\n  Check:  git -C \"{REPO}\" status     Repair:  git -C \"{REPO}\" checkout -- .   (or re-clone)")

print(f"\nRepo    : {REPO}   (found via {how_found})\nData    : {DATA}")

### A.2 Knobs and bookkeeping
**Everything you may want to change is in this one cell.** The defaults give a run of roughly half an hour on a laptop (the estimate in Part B tells you for *your* machine before anything starts). Each combination of settings gets its own folder under `compass_subset/`, so changing a knob never overwrites an earlier run, and re-running with the same settings **resumes** where an interrupted run stopped.

In [ ]:
from scipy import stats

# ---- WHICH CELLS ---------------------------------------------------------------------------------------------------------
N_PER_GROUP    = 10        # cells per group: 10 Th17p + 10 Th17n = 20 cells go through stage 2 (the expensive part)
SEED           = 0         # the cells are drawn at random with this seed (same seed -> same cells)

# ---- WHICH REACTIONS -----------------------------------------------------------------------------------------------------
REACTION_SCOPE = "fig2c"   # "fig2c": the reactions of the pathways in the paper's Fig. 2C - glycolysis, TCA cycle, fatty-acid oxidation,
                           #          amino-acid metabolism (about 2,100 of the 10,211 reactions; about 1/5 of the cost of a full cell)
                           # "all"  : every reaction, as in the paper (about 5x slower; the exchange reactions come on top)
                           # "test" : Compass' --test-mode = only the first 100 reactions (a quick trial)
                           # a path : a text file with one Recon2 reaction id per line (without _pos/_neg), e.g. PGM

# ---- HOW THE RUN IS EXECUTED ---------------------------------------------------------------------------------------------
N_PROC         = None      # stage-2 processes, one cell per process. None = all cores but one, at most 6 (each process needs about 1 GB)
MAX_HOURS      = 3         # Part C refuses to start if the estimated wall-clock time is longer than this (raise it on purpose)
WAIT_IN_NOTEBOOK = True    # True: the progress cell of Part D stays busy and refreshes every minute until Compass is finished (the square "interrupt" button only stops the watching; Compass keeps running). False: it reports once and returns
KEEP_AWAKE     = True      # macOS: keep the Mac from idle-sleeping while Compass runs (`caffeinate`); a closed lid still sleeps
GUROBI_LICENSE = "installed"  # "installed": Compass uses the Gurobi you already have installed and licensed (gurobipy finds your licence by itself). The notebook only puts an EMPTY placeholder gurobi.lic into the Compass folder because the command line insists the file exists; no secret is copied.  |  "copy": copy the licence file Gurobi uses (GRB_LICENSE_FILE, ~/gurobi.lic, ...) into the Compass folder with `compass --set-license`  |  a path: copy that file instead

# ---- HOW THE RESULT IS JUDGED --------------------------------------------------------------------------------------------
TOL_PASS  = 1e-4           # Part E: PASS if every score agrees with the authors' within this RELATIVE difference
TOL_WARN  = 1e-2           # WARN up to this, FAIL above
ALPHA     = 0.1            # [Paper Methods] a reaction is "differentially active" if its BH-adjusted p-value is below 0.1
MIN_RANGE = 1e-3           # [Paper Alg. 2 / Repo get_reaction_consistencies] reactions whose score range over the 290 cells is below this are not tested
N_DRAWS   = 200            # Part B: random subsets drawn per subset size
N_BOOT    = 1000           # Part F: bootstrap resamples for the error bars

# ---- folders and bookkeeping ---------------------------------------------------------------------------------------------
WORK = Path.cwd() / "compass_subset"
scope_tag = REACTION_SCOPE if REACTION_SCOPE in ("fig2c", "all", "test") else "custom-" + Path(str(REACTION_SCOPE)).stem
RUN = WORK / f"{2 * N_PER_GROUP}cells_seed{SEED}_{scope_tag}"      # Compass --output-dir (its temp folder is RUN/_tmp)
OUT = RUN / "results"                                              # tables and figures of Parts E-G
print(f"this experiment lives in:  {RUN}")

CHECKS = []
STATE  = {}
ICON = {"PASS": "[PASS]", "WARN": "[WARN]", "FAIL": "[FAIL]", "SKIP": "[SKIP]"}
def record(level, name, status, detail=""):
    CHECKS.append({"level": level, "check": name, "status": status, "detail": detail})
    print(f"{ICON[status]}  L{level}  {name}" + (f"  -  {detail}" if detail else ""))

### A.2b Helpers (nothing to edit here)
The file names of this experiment, a list of known error messages with advice, and two small functions that Parts C and D use. **After a kernel restart, run A.1, A.2 and this cell: Part D then works on its own** (it only reads the files Compass leaves in the experiment folder).

In [ ]:
# ---- the files of this experiment (Parts C-E use them; defined here so that Part D also works right after a kernel restart) ----
IN_DIR, LOG_DIR, TMP = RUN / "input", RUN / "logs", RUN / "_tmp"     # TMP = Compass' own working folder
DATA_FILE  = IN_DIR / "th17_expr.tsv"                # Compass --data: all 290 cells during stage 1, then ONLY the chosen cells (same path, see Part C)
SEL_FILE   = IN_DIR / "reactions_to_evaluate.txt"    # Compass --select-reactions
PREP_FILE  = RUN / "prepared.json"                   # written when stage 1 and the penalty remapping are finished and verified
STATE_FILE = RUN / "run_state.json"                  # written when stage 2 is launched (process id, start time, ...)

# ---- known error messages and what to do about them (used by Parts C and D) ----
SIGNATURES = [
    ("a gurobi license is required", "Compass wants a gurobi.lic file inside its own folder. Re-run the pre-flight (A.3) with GUROBI_LICENSE = \"installed\", which creates an empty placeholder."),
    ("too large for size-limited license", "Size-limited Gurobi licence (max 2,000 variables). Recon2 needs the full academic licence."),
    ("no gurobi license found", "Gurobi cannot find a licence by itself: put gurobi.lic in your home folder (~/gurobi.lic) or set GRB_LICENSE_FILE in the environment that starts the notebook."),
    ("expired", "A licence has expired - renew it in the Gurobi user portal (academic licences last one year)."),
    ("eoferror", "Compass found that its arguments differ from those of the earlier stage-1 run and asked a question. Delete the experiment folder (shown above) and run Part C again."),
    ("modulenotfounderror", "A Python package is missing in the environment that runs Compass (the module name is in the error text)."),
    ("memoryerror", "Out of memory - set N_PROC lower (each process needs about 1 GB) and run the launch cell again; finished cells are kept."),
    ("killed", "The process was killed from outside, most likely by the operating system for lack of memory. Set N_PROC lower and run the launch cell again; finished cells are kept."),
]
def diagnose(text):
    t = text.lower()
    tips = [tip for key, tip in SIGNATURES if key in t]
    if not tips and "licen" in t:
        tips = ["A Gurobi licence problem (see the pre-flight, A.3). If the message mentions a session or usage limit, lower N_PROC (try 3, then 1) and run the launch cell again; finished cells are kept."]
    return tips

def explain_failure(headline):
    """Print the last meaningful lines of every Compass log, with the matching advice from SIGNATURES."""
    txt = ""
    for f in [*sorted(LOG_DIR.glob("*.txt")), *sorted(RUN.glob("_tmp/sample*/err.log"))]:
        txt += f.read_text(errors="replace") + "\n"
    tail = [l for l in txt.replace("\r", "\n").splitlines() if l.strip() and "it/s" not in l and "s/it" not in l and "%|" not in l][-8:]
    print(headline)
    for tip in diagnose(txt):
        print("   advice:", tip)
    if tail:
        print("   last lines of the error output:\n     " + "\n     ".join(l[:170] for l in tail))
    print(f"   all logs: {LOG_DIR}   and   {RUN / '_tmp'}/sample*/err.log")

def read_run_state():
    """What the launch cell wrote down (process id, start time, ...), or None if stage 2 was never launched."""
    return json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else None

def pid_alive(pid):
    """True if the process exists, is not a zombie (a finished child nobody has collected yet) and is a Compass process."""
    r = subprocess.run(["ps", "-o", "stat=,command=", "-p", str(pid)], capture_output=True, text=True)
    line = r.stdout.strip()
    return bool(line) and not line.startswith("Z") and "compass" in line.lower()

### A.3 Pre-flight: the first three checks of Notebook 4, unchanged
These are the same cells as Notebook 4, levels 1–3: (1) packages and the `compass` command, (2) the Recon2 model and its cached `v_opt` load, (3) Gurobi works and its licence is **full-size**. Everything after this depends on them; if one fails, the last cell of this section tells you what to fix.

In [ ]:
import importlib, tempfile
from importlib.metadata import version, PackageNotFoundError

NEEDED = [("numpy", "numpy"), ("pandas", "pandas"), ("scipy", "scipy"), ("sklearn", "scikit-learn"), ("tqdm", "tqdm"),
          ("six", "six"), ("libsbml", "python-libsbml"), ("igraph", "python-igraph"), ("leidenalg", "leidenalg"),
          ("anndata", "anndata"), ("gurobipy", "gurobipy"), ("session_info", "session-info")]

def probe_compass(extra_env=None):
    """Import compass in a NEW Python process started in a neutral folder; return (info, error_line)."""
    probe = ("import json, compass, compass.globals as g\n"
             "print(json.dumps({'file': compass.__file__, 'license_dir': g.LICENSE_DIR}))")
    r = subprocess.run([sys.executable, "-c", probe], cwd=tempfile.gettempdir(), env={**os.environ, **(extra_env or {})},
                       capture_output=True, text=True)
    if r.returncode == 0:
        return json.loads(r.stdout.strip().splitlines()[-1]), None
    return None, (r.stderr.strip().splitlines() or ["unknown error"])[-1]

def level1():
    # (a) packages -------------------------------------------------------------------------------------------
    rows, bad = [], []
    for mod, pipname in NEEDED:
        try:
            importlib.import_module(mod)
            try: v = version(pipname)
            except PackageNotFoundError: v = "?"
            rows.append((pipname, v, "ok"))
        except Exception as e:                          # missing package OR a compiled library that fails to load
            rows.append((pipname, "-", f"{type(e).__name__}: {str(e)[:70]}")); bad.append(pipname)
    print(pd.DataFrame(rows, columns=["package", "version", "import"]).to_string(index=False), "\n")
    if bad:
        record(1, "packages Compass needs at start-up", "FAIL",
               f"missing or broken: {', '.join(bad)}.  Fix:  \"{sys.executable}\" -m pip install {' '.join(bad)}")
        return
    record(1, "packages Compass needs at start-up", "PASS", f"all {len(NEEDED)} import")

    # (b) where does `compass` come from? ---------------------------------------------------------------------
    env = dict(os.environ)
    info, err = probe_compass()
    if info is None:
        info2, err2 = probe_compass({"PYTHONPATH": str(REPO)})
        if info2 is None:
            record(1, "import compass", "FAIL", f"{err2}   (repo: {REPO})")
            return
        env["PYTHONPATH"] = str(REPO) + (os.pathsep + os.environ["PYTHONPATH"] if os.environ.get("PYTHONPATH") else "")
        info = info2
        record(1, "import compass", "WARN",
               f"not installed in this Python; using your clone through PYTHONPATH (fine for this test). For daily use run:  "
               f"\"{sys.executable}\" -m pip install -e \"{REPO}\"")
    else:
        where = Path(info["file"]).resolve()
        if REPO in where.parents:
            record(1, "import compass", "PASS", f"installed from your clone: {where.parent}")
        else:
            record(1, "import compass", "WARN",
                   f"a separate copy in {where.parent} is used, NOT your clone - edits to the clone will not affect it")
    STATE["env"] = env
    STATE["license_dir"] = Path(info["license_dir"])
    STATE["lic_file"] = STATE["license_dir"] / "gurobi.lic"

    # (c) how do we start the command line? --------------------------------------------------------------------
    script = Path(sys.executable).parent / ("compass.exe" if os.name == "nt" else "compass")
    if script.exists():
        STATE["cmd"] = [str(script)]
        record(1, "`compass` command", "PASS", f"console script next to this Python: {script}")
    else:
        STATE["cmd"] = [sys.executable, "-c", "from compass.main import entry; entry()"]
        record(1, "`compass` command", "WARN", "no console script next to this Python; starting it with `python -c` instead (equivalent)")
    STATE["level1_ok"] = True

level1()
if STATE.get("env") is not None:
    print("\nCompass folder (where it looks for gurobi.lic):", STATE["license_dir"])

In [ ]:
def level2():
    if not STATE.get("level1_ok"):
        return record(2, "model + cache", "SKIP", "level 1 did not pass")
    if STATE["env"].get("PYTHONPATH"):
        sys.path.insert(0, STATE["env"]["PYTHONPATH"].split(os.pathsep)[0])
    try:
        from compass import models as cmodels, globals as cglobals
        from compass.compass import cache as ccache
        cm = cmodels.init_model("RECON2_mat", species="mus_musculus", exchange_limit=cglobals.EXCHANGE_LIMIT,
                                media="default-media", isoform_summing="legacy")
    except Exception as e:
        return record(2, "model loads through the repo's code", "FAIL", f"{type(e).__name__}: {str(e)[:200]}")
    STATE.update(cm=cm, ccache=ccache, BETA=cglobals.BETA)
    ref = pd.read_csv(DATA / "Th17/reactions.tsv", sep="\t", index_col=0)           # the authors' output
    STATE["ref"] = ref
    n_rxn, n_met = len(cm.reactions), len(cm.species)
    print(f"model: {n_rxn:,} unidirectional reactions, {n_met:,} metabolites, species mus_musculus, medium default-media")
    same = set(cm.reactions) == set(ref.index)
    record(2, "model loads; reaction set equals the authors' output", "PASS" if same else "FAIL",
           f"{n_rxn:,} reactions" + ("" if same else f" vs {len(ref):,} in the authors' reactions.tsv"))
    vopt = ccache.load(cm)
    STATE["vopt"] = vopt
    covered = set(vopt) >= set(cm.reactions)
    zero = float(np.mean([vopt[r] == 0 for r in cm.reactions if r in vopt]))
    print(f"cached v_opt: {len(vopt):,} entries; {zero:.0%} are 0 (reactions that can never carry flux; paper: 33% blocked)")
    record(2, "cached v_opt covers every reaction", "PASS" if covered else "FAIL", f"{len(vopt):,} entries")
    STATE["level2_ok"] = same and covered

level2()

In [ ]:
def find_gurobi_license_file():
    """The licence FILE Gurobi itself uses on this computer, if there is one (the places Gurobi searches by default)."""
    for c in [os.environ.get("GRB_LICENSE_FILE"), Path.home() / "gurobi.lic", "/Library/gurobi/gurobi.lic",
              "/opt/gurobi/gurobi.lic", Path.home() / "gurobi" / "gurobi.lic"]:
        if c and Path(c).expanduser().is_file():
            return Path(c).expanduser()
    return None

def parse_license_like_compass(path):
    """Read a licence file exactly the way Compass does (utils.parse_gurobi_license_file): every line containing '=' is split
    into KEY and VALUE; a line with TWO or more '=' makes `key, val = line.split('=')` crash. Uses Compass' own function when
    it can be imported, otherwise an identical copy of its logic."""
    try:
        from compass import utils as cutils
        return cutils.parse_gurobi_license_file(str(path))
    except ImportError:
        creds = {}
        with open(path, "r") as fh:
            for line in fh:
                line = line.strip()
                if "=" in line:
                    key, val = line.split("=")
                    try: val = int(val)
                    except ValueError: pass
                    creds[key] = val
        return creds

PLACEHOLDER_TEXT = (
    "# Placeholder written by 04_Compass_trial_run_validity_check.ipynb\n"
    "# Compass only checks that this file exists. It holds no credentials, so Compass uses the Gurobi licence that gurobipy finds by itself.\n")

def level3():
    if not STATE.get("level1_ok"):
        return record(3, "Gurobi licence", "SKIP", "level 1 did not pass")
    lic = STATE["lic_file"]
    found = find_gurobi_license_file()

    # (a) the file the command line insists on --------------------------------------------------------------------
    # main.py: entry() quits unless <Compass folder>/gurobi.lic exists. algorithm.py then reads WLS keys from it if all three are
    # present and otherwise calls gp.Env() = Gurobi's own licence lookup (the Gurobi you installed).
    mode = GUROBI_LICENSE
    if not lic.exists():
        if mode == "installed":
            try:
                lic.write_text(PLACEHOLDER_TEXT)
                print(f"The Compass folder had no gurobi.lic. Wrote an EMPTY placeholder (no credentials) at {lic}\n"
                      "   -> Compass will use the Gurobi you installed and licensed (gp.Env() finds your licence by itself).")
            except OSError as e:
                print(f"Could not write the placeholder: {type(e).__name__}: {e}")
        elif mode:  # "copy" or a path: copy a real licence file with Compass' own documented command
            src = found if mode == "copy" else Path(mode).expanduser()
            if src is None:
                print("The Compass folder has no gurobi.lic and no licence file was found in the usual places "
                      "(GRB_LICENSE_FILE, ~/gurobi.lic, /Library/gurobi, /opt/gurobi) - nothing to copy.")
            else:
                print(f"The Compass folder has no gurobi.lic. Copying  {src}  into it with Compass' own documented command (compass --set-license) ...")
                r = subprocess.run(STATE["cmd"] + ["--set-license", str(src)], cwd=tempfile.gettempdir(), env=STATE["env"],
                                   capture_output=True, text=True, stdin=subprocess.DEVNULL)
                print("   ->", (r.stdout or r.stderr).strip()[:200])

    creds = {}
    if lic.exists():
        try:
            creds = parse_license_like_compass(lic)
            if all(k in creds for k in ("WLSACCESSID", "WLSSECRET", "LICENSEID")):
                kind = "contains WLS credentials: Compass uses THESE"
            elif creds:
                kind = "keys but no WLS credentials: Compass ignores them and uses Gurobi's own licence lookup"
            else:
                kind = "no credentials: Compass uses the Gurobi you installed (Gurobi's own licence lookup)"
            record(3, "gurobi.lic in the Compass folder (needed by the command line)", "PASS", f"{lic}  [{kind}]")
        except Exception as e:
            creds = {}
            record(3, "gurobi.lic in the Compass folder can be read by Compass", "FAIL",
                   f"{type(e).__name__}: {str(e)[:120]}  (Compass reads this file line by line as KEY=VALUE; stage 2 would crash on it). "
                   "Fix: delete that file and set GUROBI_LICENSE = \"installed\" in cell 0.2")
        if creds and (REPO / ".git").exists() and REPO in lic.resolve().parents:
            print("   note: this file lives INSIDE your git clone, holds licence keys and the repo's .gitignore does not exclude it - never `git add` it.")
    else:
        record(3, "gurobi.lic in the Compass folder (needed by the command line)", "FAIL",
               f"not found at {lic}.  Fix: set GUROBI_LICENSE = \"installed\" in cell 0.2 (creates an empty placeholder), "
               "or once in a terminal:  compass --set-license ~/gurobi.lic")
    STATE["lic_ok"] = lic.exists()

    # (b) what does Gurobi itself allow? (same environment logic as opt/gurobi.py) ----------------------------------------
    try:
        import gurobipy as gp
        from gurobipy import GRB
        env = gp.Env(empty=True); env.setParam("OutputFlag", 0)
        if all(k in creds for k in ("WLSACCESSID", "WLSSECRET", "LICENSEID")):
            for k in ("WLSACCESSID", "WLSSECRET", "LICENSEID"): env.setParam(k, creds[k])
        env.start()
        m = gp.Model(env=env); x = m.addVars(2, lb=0, ub=10); m.addConstr(x[0] + x[1] >= 1); m.setObjective(x[0] + x[1], GRB.MINIMIZE); m.optimize()
        assert m.Status == GRB.OPTIMAL and abs(m.ObjVal - 1.0) < 1e-9
        v = gp.gurobi.version()
        record(3, "Gurobi starts and solves a tiny LP", "PASS", f"gurobipy {v[0]}.{v[1]}.{v[2]}")
    except Exception as e:
        STATE["gurobi"] = "NONE"
        return record(3, "Gurobi starts and solves a tiny LP", "FAIL", f"{type(e).__name__}: {str(e)[:160]}")
    try:
        m = gp.Model(env=env); y = m.addVars(2100, lb=0, ub=1); m.addConstr(y.sum() >= 1); m.setObjective(y.sum(), GRB.MINIMIZE); m.optimize()
        STATE["gurobi"] = "FULL"
        record(3, "licence size", "PASS", "FULL - solved a 2,100-variable LP, so Recon2 (10,211 variables) is allowed")
    except gp.GurobiError as e:
        if "too large" in str(e).lower():
            STATE["gurobi"] = "RESTRICTED"
            record(3, "licence size", "FAIL",
                   "SIZE-LIMITED licence (max 2,000 variables). Compass on Recon2 needs a full licence: get the free academic (WLS) licence "
                   "via gurobi.com/unrestricted and save the file as ~/gurobi.lic (the installed Gurobi finds it by itself), then re-run from Level 3")
        else:
            STATE["gurobi"] = "NONE"
            record(3, "licence size", "FAIL", f"{str(e)[:160]}")

level3()

In [ ]:
bad = [c for c in CHECKS if c["status"] == "FAIL"]
preflight_ok = bool(STATE.get("level1_ok") and STATE.get("level2_ok") and STATE.get("lic_ok") and STATE.get("gurobi") == "FULL" and not bad)
if preflight_ok:
    print("PRE-FLIGHT OK - Compass starts, the model loads, and Gurobi has a full-size licence. You can go on.")
else:
    print("PRE-FLIGHT NOT PASSED - fix the first [FAIL] above (see Notebook 4 and README_SETUP.md section 2.8), then re-run from that cell.")
    for c in bad[:3]:
        print(f"   [FAIL] L{c['level']} {c['check']}: {c['detail']}")

## Part B · Design the experiment (no Compass needed yet)
### B.0 Statistics helpers (copied unchanged from Notebook 1, section 6)
Three small functions used twice below — once on the authors' output, once on yours — so both go through exactly the same code.
* **Cohen's d** = (mean of group A − mean of group B) / pooled standard deviation: *how many standard deviations apart the two groups are*. Positive = higher in A (here Th17p). It measures **size**, not certainty.
* **Wilcoxon rank-sum (Mann–Whitney U)**: a test of whether one group's values tend to be larger, using only the *ranks* of the values; it does not assume a bell curve.
* **Benjamini–Hochberg (BH)**: with thousands of tests some will look significant by luck; BH rescales the p-values so that "adjusted p < 0.1" means roughly "at most 10 % of the reactions I call significant are false alarms".

In [ ]:
def cohens_d_rows(A, B):
    """Cohen's d per row = (mean_A - mean_B) / pooled SD.   [Paper STAR Methods; Repo compass_analysis.cohens_d]"""
    nA, nB = A.shape[1], B.shape[1]
    pooled = np.sqrt(((nA - 1) * A.var(axis=1, ddof=1) + (nB - 1) * B.var(axis=1, ddof=1)) / (nA + nB - 2))
    with np.errstate(divide="ignore", invalid="ignore"):
        return (A.mean(axis=1) - B.mean(axis=1)) / pooled

def benjamini_hochberg(p):
    """Benjamini-Hochberg adjusted p-values (FDR).
       1) sort the m p-values ascending; 2) adjusted(i) = p(i) * m / i  (i = rank);
       3) go from the largest rank down and keep the running minimum (adjusted values never decrease with p); 4) cap at 1."""
    p = np.asarray(p, float)
    m = len(p)
    order = np.argsort(p)
    scaled = p[order] * m / np.arange(1, m + 1)
    adj_sorted = np.minimum.accumulate(scaled[::-1])[::-1]
    adj = np.empty(m)
    adj[order] = np.clip(adj_sorted, 0, 1)
    return adj

def differential_activity(scores, cells_a, cells_b):
    """Wilcoxon rank-sum (Mann-Whitney, two-sided) + Cohen's d + Benjamini-Hochberg, one row per reaction.
       Positive d  =>  higher potential activity in group A.   [Paper 'Finding reactions with differential potential activity']"""
    A, B = scores[cells_a].to_numpy(float), scores[cells_b].to_numpy(float)
    res = stats.mannwhitneyu(A, B, axis=1, alternative="two-sided")
    p = np.where(np.isnan(res.pvalue), 1.0, res.pvalue)               # constant rows -> p = 1 (as in the repo)
    out = pd.DataFrame({"U": res.statistic, "p": p, "cohens_d": cohens_d_rows(A, B)}, index=scores.index)
    out["adj_p"] = benjamini_hochberg(out["p"])
    return out

def base_id(r):
    """'PGM_neg' -> 'PGM' (the Recon2 metadata table has no direction suffix)"""
    return r[:-4] if r.endswith(("_pos", "_neg")) else r

### B.1 Which cells?
We draw **`N_PER_GROUP` Th17p and `N_PER_GROUP` Th17n cells at random** (fixed `SEED`, so the choice is reproducible). Random draws are unbiased: choosing "typical-looking" cells would hide exactly the cell-to-cell variation the analysis is about. The notebook records each cell's **position (0-based) in the 290-column file** — that position is what Compass uses internally (see the trap in the introduction).

In [ ]:
if "ref" not in STATE:
    raise RuntimeError("Run the pre-flight cells in Part A first (they load the model and the authors' output).")

# ---- the four panels of the paper's Fig. 2C (same lists as Notebook 1, section 6.5) ----------------------------------------------
AMINO_ACID = ["Alanine and aspartate metabolism", "Arginine and Proline Metabolism", "beta-Alanine metabolism", "Cysteine Metabolism",
              "D-alanine metabolism", "Folate metabolism", "Glutamate metabolism", "Glycine, serine, alanine and threonine metabolism",
              "Histidine metabolism", "Lysine metabolism", "Methionine and cysteine metabolism", "Taurine and hypotaurine metabolism",
              "Tryptophan metabolism", "Tyrosine metabolism", "Urea cycle", "Valine, leucine, and isoleucine metabolism"]   # [Repo]
PANELS = {"Glycolysis": ["Glycolysis/gluconeogenesis"], "TCA cycle": ["Citric acid cycle"],
          "Fatty-acid oxidation": ["Fatty acid oxidation"], "Amino-acid metabolism": AMINO_ACID}
PANEL_COLOUR = {"Glycolysis": "#695D73", "TCA cycle": "#D3A991", "Fatty-acid oxidation": "#040772", "Amino-acid metabolism": "#BF1E2E", "other": "lightgrey"}
LABELED = {   # [Repo compass_analysis.py `labeled_reactions`] the reactions the authors annotate in Fig. 2C
    "PGM_neg": "phosphoglycerate mutase", "LDH_L_neg": "lactate dehydrogenase", "PDHm_pos": "pyruvate dehydrogenase",
    "TPI_neg": "triosephosphate isomerase", "FACOAL1821_neg": "long-chain FA-CoA ligase", "r1257_pos": "long-chain FA-CoA ligase",
    "FACOAL1831_neg": "long-chain FA-CoA ligase", "CSNATr_neg": "carnitine O-acetyltransferase", "C160CPT1_pos": "carnitine palmitoyltransferase",
    "ACONTm_pos": "aconitate hydratase", "SUCOASm_pos": "succinate-CoA ligase", "AKGDm_pos": "alpha-KG dehydrogenase",
    "SUCD1m_pos": "succinate dehydrogenase", "ICDHyrm_pos": "isocitrate dehydrogenase", "CK_pos": "creatine kinase",
    "PGCD_pos": "phosphoglycerate dehydrogenase", "ARGSS_pos": "arginosuccinate synthase", "r0281_neg": "putrescine diamine oxidase",
    "SPMDOX_pos": "spermidine dehydrogenase", "ARGDCm_pos": "arginine decarboxylase", "AGMTm_pos": "agmatinase",
    "GHMT2r_pos": "serine hydroxymethyltransferase", "AHC_pos": "adenosylhomocysteinase", "METAT_pos": "methionine adenosyltransferase",
    "METS_pos": "methionine synthase", "ARGN_pos": "arginase",
}
PAL = {"Th17p": "#E92E87", "Th17n": "#348C73"}

# ---- tables ---------------------------------------------------------------------------------------------------------------------
expr_cols = list(pd.read_csv(DATA / "Th17/linear_gene_expression_matrix.tsv", sep="\t", index_col=0, nrows=1).columns)
meta    = pd.read_csv(DATA / "Th17/cell_metadata.csv", index_col=0)
rxn_md  = pd.read_csv(DATA / "RECON2/reaction_metadata.csv", index_col=0)
pen_all = STATE["ref"]                                                    # the authors' Compass output: reactions x 290 cells (penalties)
assert expr_cols == list(meta.index) == list(pen_all.columns), "the expression matrix, the metadata and the authors' output list the cells in different orders"
group = meta.cell_type
p_idx, n_idx = np.flatnonzero((group == "Th17p").to_numpy()), np.flatnonzero((group == "Th17n").to_numpy())
if N_PER_GROUP > min(len(p_idx), len(n_idx)):
    raise ValueError(f"N_PER_GROUP = {N_PER_GROUP} is larger than the smaller group ({min(len(p_idx), len(n_idx))} cells)")

# ---- the draw -------------------------------------------------------------------------------------------------------------------
rng = np.random.default_rng(SEED)
positions = sorted(rng.choice(p_idx, N_PER_GROUP, replace=False).tolist() + rng.choice(n_idx, N_PER_GROUP, replace=False).tolist())
cells = [expr_cols[i] for i in positions]
cells_p_sub = [c for c in cells if group[c] == "Th17p"]
cells_n_sub = [c for c in cells if group[c] == "Th17n"]
print(f"{len(cells)} cells chosen with seed {SEED}:  {len(cells_p_sub)} Th17p + {len(cells_n_sub)} Th17n   (of {len(p_idx)} + {len(n_idx)})")
display(pd.DataFrame({"cell": cells, "group": [group[c] for c in cells], "position in the 290-column file": positions}))

### B.2 Which reactions, and what will it cost?
A cell's expensive work is **one linear program (LP) per reaction** whose cached maximal flux `v_opt` is > 0 (the paper: 33 % of the reactions have `v_opt = 0` and cost nothing) **[Paper]**. Compass can restrict the loop to selected reactions with `--select-reactions` — the model stays the full Recon2 network, only the list of reactions to *evaluate* is shorter, so each score is exactly the number a full run would give **[Repo, algorithm.py `compass_reactions`: the selection filters the loop only]**. Exchange reactions (uptake/secretion) are evaluated only if they are selected; with `fig2c` they are not, which saves a large block of slower LPs.

`REACTION_SCOPE = "fig2c"` selects every reaction of the four pathways shown in the paper's Fig. 2C (glycolysis, TCA cycle, fatty-acid oxidation, amino-acid metabolism) **plus** the 26 reactions the authors label there. The cell below counts the LPs with **Compass' own selection rule** and turns the count into a time estimate.

**Where the speed comes from.** If Notebook 4's trial left its logs in `compass_trial/`, the notebook measures *your* seconds per LP there. Otherwise it uses the repo's documentation — "roughly 30 minutes per sample" for a full cell **[Repo, docs/quickstart.rst]** — divided by the number of LPs of a full cell. Either way it is **[Inference]**: an order of magnitude, not a promise.

In [ ]:
cm, vopt = STATE["cm"], STATE["vopt"]
all_rx = list(cm.reactions.values())

# ---- the list of reactions to evaluate -----------------------------------------------------------------------------------------
if REACTION_SCOPE == "fig2c":
    subs = [s for v in PANELS.values() for s in v]
    SCOPE_IDS = set(rxn_md.index[rxn_md.subsystem.isin(subs)]) | {base_id(r) for r in LABELED}
elif REACTION_SCOPE in ("all", "test"):
    SCOPE_IDS = None
else:
    SCOPE_IDS = {l.strip() for l in Path(str(REACTION_SCOPE)).expanduser().read_text().splitlines() if l.strip()}
if SCOPE_IDS is not None:
    unknown = sorted(i for i in SCOPE_IDS if i not in cm.reactions and i + "_pos" not in cm.reactions and i + "_neg" not in cm.reactions)
    if unknown:
        print(f"note: {len(unknown)} id(s) of the pathway tables are not reactions of the model used here; Compass ignores them (e.g. {unknown[:3]})")

if REACTION_SCOPE == "test":   evaluated = all_rx[:100]                                  # Compass --test-mode: the first 100 reactions
elif REACTION_SCOPE == "all":  evaluated = all_rx
else:                          evaluated = [r for r in all_rx if str(r.id)[:-4] in SCOPE_IDS or str(r.id) in SCOPE_IDS]   # Compass' own rule
need_lp = [r for r in evaluated if not r.is_exchange and vopt.get(r.id, 0) > 0]          # reactions that need an LP (v_opt = 0 -> score 0, no LP)

n_lp_full = sum(1 for r in all_rx if not r.is_exchange and vopt.get(r.id, 0) > 0)         # LPs of a complete cell, exchange reactions excluded
exch_mets = set()
for r in all_rx:
    if r.is_exchange:
        exch_mets.update(r.reactants); exch_mets.update(r.products)
n_exch_lp = 4 * len(exch_mets)               # [Inference] per metabolite with an exchange reaction: 2 LPs for uptake + 2 for secretion (algorithm.py compass_exchange)
lp_per_cell = len(need_lp) + (n_exch_lp if REACTION_SCOPE == "all" else 0)

# ---- seconds per LP -----------------------------------------------------------------------------------------------------------------
def trial_seconds_per_lp():
    """Measured in Notebook 4's trial: CPU seconds of the reaction loop / number of LPs it needed."""
    logs = sorted((Path.cwd() / "compass_trial" / "run" / "_tmp").glob("sample*/compass.log"))
    n_lp = sum(1 for r in all_rx[:100] if not r.is_exchange and vopt.get(r.id, 0) > 0)       # the trial used --test-mode = first 100 reactions
    cpu = []
    for f in logs:
        m = re.search(r"Compass Reaction Time:\s*([\d.]+)", f.read_text(errors="replace"))
        if m: cpu.append(float(m.group(1)))
    return (float(np.mean(cpu)) / n_lp, len(cpu), n_lp) if cpu and n_lp else None

meas = trial_seconds_per_lp()
if meas:
    sec_per_lp, speed_src = meas[0], f"measured in your Notebook 4 trial ({meas[1]} cells, {meas[2]} LPs each)"
else:
    sec_per_lp, speed_src = 1800 / n_lp_full, "repo documentation (about 30 min per full cell / %s LPs); run Notebook 4 to measure your machine" % f"{n_lp_full:,}"

N_PROC_USED = N_PROC or max(1, min(len(cells), (os.cpu_count() or 2) - 1, 6))
cpu_h  = len(cells) * lp_per_cell * sec_per_lp / 3600
wall_h = cpu_h / min(N_PROC_USED, len(cells))
STATE["est_hours"] = wall_h

print(f"cells                    : {len(cells)}  ({len(cells_p_sub)} Th17p + {len(cells_n_sub)} Th17n)")
print(f"reactions evaluated      : {len(evaluated):,} of {len(all_rx):,}   ->  {len(need_lp):,} need an LP per cell   (a complete cell: {n_lp_full:,}; plus about {n_exch_lp:,} for the exchange reactions)")
print(f"speed                    : {sec_per_lp:.2f} s per LP   [{speed_src}]")
print(f"estimated compute        : {cpu_h:.1f} CPU-hours  ->  about {wall_h * 60:.0f} minutes ({wall_h:.1f} h) on {min(N_PROC_USED, len(cells))} processes   [Inference, rough]")
if wall_h > MAX_HOURS:
    print(f"! longer than MAX_HOURS = {MAX_HOURS}: Part C will refuse to start. Reduce N_PER_GROUP / REACTION_SCOPE, or raise MAX_HOURS on purpose.")
else:
    print(f"within MAX_HOURS = {MAX_HOURS}: OK to start.")

### B.3 How many cells are enough? A simulation on the authors' own numbers
**Question.** With only 10 + 10 cells, how closely can the *size* of a reaction's difference (Cohen's d) be estimated, compared with using all 290 cells? We can answer this **without running Compass**, because the authors' `reactions.tsv` already contains every cell's score.

**Method.** For each subset size *n* (n Th17p + n Th17n), draw `N_DRAWS` random subsets, compute Cohen's d per reaction from the authors' scores, and measure the **Spearman rank correlation** (does the subset rank the reactions in the same order?) with the d computed from all 290 cells. A correlation of 1 means the subset orders the reactions exactly as the full data do; 0 means it carries no information. Only the reactions in your chosen scope are used.

**Caveat [Inference].** The 290 cells contain the subset, so the two estimates are not independent; the simulation is mildly optimistic, increasingly so for large *n*. After your run, Part F puts **your** correlation next to this distribution.

In [ ]:
S_auth = -np.log(pen_all + 1.0)                                        # score = -log(1 + penalty)   [Paper Alg. 3; Repo get_reaction_consistencies]
kept = (S_auth.max(axis=1) - S_auth.min(axis=1)) >= MIN_RANGE          # reactions the paper's analysis keeps (not constant over the 290 cells)
rows_B = [r.id for r in evaluated if kept.get(r.id, False)]
A_all = S_auth.loc[rows_B].to_numpy(float)
d_full = cohens_d_rows(A_all[:, p_idx], A_all[:, n_idx])               # Cohen's d from ALL 290 cells (the reference)

def baseline_rho(ns, draws, seed=12345):
    r = np.random.default_rng(seed); out = []
    for n in ns:
        for _ in range(draws):
            d = cohens_d_rows(A_all[:, r.choice(p_idx, n, replace=False)], A_all[:, r.choice(n_idx, n, replace=False)])
            out.append((n, pd.Series(d).corr(pd.Series(d_full), method="spearman")))
    return pd.DataFrame(out, columns=["n_per_group", "rho"])

ns = sorted({n for n in (3, 5, 10, 20, 40, N_PER_GROUP) if n <= min(len(p_idx), len(n_idx))})
baseline = baseline_rho(ns, N_DRAWS)
STATE["baseline"] = baseline

def plot_baseline(ax, mark=None):
    data = [baseline.loc[baseline.n_per_group == n, "rho"].dropna() for n in ns]
    bp = ax.boxplot(data, showfliers=False, patch_artist=True)
    for patch, n in zip(bp["boxes"], ns):
        patch.set_facecolor("#E92E87" if n == N_PER_GROUP else "lightgrey")
    ax.set_xticks(range(1, len(ns) + 1)); ax.set_xticklabels([f"{n}+{n}" for n in ns])
    ax.set_xlabel("cells per group (Th17p + Th17n)"); ax.set_ylabel("Spearman correlation of Cohen's d\n(subset vs all 290 cells)"); ax.set_ylim(0, 1.02)
    if mark is not None:
        k = ns.index(N_PER_GROUP) + 1
        ax.scatter([k], [mark], marker="*", s=220, color="black", zorder=5, label="YOUR run"); ax.legend(frameon=False, loc="lower right")

fig, ax = plt.subplots(figsize=(6.4, 4))
plot_baseline(ax); ax.set_title(f"Expected agreement with the full data ({len(rows_B):,} reactions in scope, {N_DRAWS} random subsets each)")
ax.title.set_fontsize(9); plt.tight_layout(); plt.show()
summ = baseline.groupby("n_per_group")["rho"].quantile([.1, .5, .9]).unstack().round(2)
summ.columns = ["10th percentile", "median", "90th percentile"]; summ.index.name = "cells per group"
display(summ)
m = float(summ.loc[N_PER_GROUP, "median"])
print(f"With {N_PER_GROUP}+{N_PER_GROUP} cells the median rank correlation is {m:.2f}: the order of the reactions' effect sizes is "
      + ("well recovered, but individual reactions remain noisy." if m >= 0.7 else "only roughly recovered - treat single-reaction results with caution."))

**How to read the figure.** Each box shows where the middle half of the `N_DRAWS` random subsets falls (line = median, whiskers = the rest, outliers hidden); the pink box is your choice. The boxes climb with *n*, but with diminishing returns — doubling the cells helps less each time. If the pink box sits well below 0.7, expect your subset to *roughly* reproduce the main pattern (for example "glycolysis is higher in Th17p") but not the ranking of individual reactions; the table under the figure gives the 10th–90th percentile range so you can see how much luck is involved.

## Part C · Run Compass
Two steps, each in its own cell. Both are **safe to run again**: nothing that is already done is repeated.

| Step | What it does | Time |
|---|---|---|
| **C.2 prepare** | **Stage 1** on all 290 cells (the penalties), then copies the penalties of your 20 cells to the places stage 2 will look for them, **verifies by cell name** that every copy is right, and shrinks the data file to your 20 cells | ≈ 0.5–2 min |
| **C.3 launch** | **Stage 2** (the linear programs) as a **background process** — the notebook is free again immediately, and you may close it | instant |

### C.1 How the command line is built
Both stages use **the same options** — the paper's exact ones **[Paper, STAR Methods; README_SETUP.md §2.6]**: `--model RECON2_mat --species mus_musculus --media default-media --lambda 0.25 --and-function mean --penalty-diffusion knn --num-neighbors 10 --isoform-summing legacy`. They must be identical, because Compass stores the options of the first run in `_tmp/_temp_args.json` and asks a question when a later run differs **[Repo, main.py `entry`]**. The only difference between the stages is `--only-penalties` (stage 1) and `--num-processes`, which Compass ignores in that comparison.

In [ ]:
def compass_args():
    """The Compass command line, identical in stage 1 and stage 2 (see the text above)."""
    a = ["--data", str(DATA_FILE), "--model", "RECON2_mat", "--species", "mus_musculus", "--media", "default-media",
         "--lambda", "0.25", "--and-function", "mean", "--penalty-diffusion", "knn", "--num-neighbors", "10",
         "--isoform-summing", "legacy", "--output-dir", str(RUN)]
    if REACTION_SCOPE == "test":
        a += ["--test-mode"]                                      # the first 100 reactions
    elif REACTION_SCOPE != "all":
        a += ["--select-reactions", str(SEL_FILE)]                # only the reactions of the chosen scope
    return a

def run_blocking(args, label, timeout_min=30):
    """Run Compass as a separate process and wait for it (used for stage 1 only). Console output goes to RUN/logs/<label>.*.txt"""
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    t0, last = time.time(), 0
    with open(LOG_DIR / f"{label}.out.txt", "w") as fo, open(LOG_DIR / f"{label}.err.txt", "w") as fe:
        p = subprocess.Popen(STATE["cmd"] + args, cwd=RUN, env={**STATE["env"], "PYTHONUNBUFFERED": "1"},
                             stdout=fo, stderr=fe, stdin=subprocess.DEVNULL)
        try:
            while p.poll() is None:
                time.sleep(1)
                el = time.time() - t0
                if el - last >= 20:
                    print(f"   ... {el:4.0f} s elapsed"); last = el
                if el > timeout_min * 60:
                    p.kill(); p.wait()
                    return {"code": None, "secs": el, "timeout": True}
        except KeyboardInterrupt:                                 # you pressed "interrupt" in the notebook: do not leave Compass running
            p.kill(); p.wait(); raise
    return {"code": p.returncode, "secs": time.time() - t0, "timeout": False}

def prepare():
    """Stage 1 on all cells -> penalties of the chosen cells moved to positions 0..N-1 (verified by name) -> data file = chosen cells."""
    if PREP_FILE.exists():
        prep = json.loads(PREP_FILE.read_text())
        if prep["cells"] != cells:
            print(f"! {RUN}\n  was prepared for OTHER cells. Delete that folder (or change SEED / N_PER_GROUP / REACTION_SCOPE) and run this cell again.")
            return False
        print(f"already prepared ({PREP_FILE.name}: stage 1 done, penalties placed and verified) - nothing to do.")
        return True
    if STATE_FILE.exists():
        print(f"! {RUN} holds a launched run but no {PREP_FILE.name}. Refusing to wipe it; delete the folder by hand if you want to start over.")
        return False
    if RUN.exists():                                              # a half-finished earlier preparation: nothing of value in it yet
        print("removing a half-finished earlier preparation:", RUN)
        shutil.rmtree(RUN)
    IN_DIR.mkdir(parents=True); LOG_DIR.mkdir(parents=True)
    shutil.copyfile(DATA / "Th17/linear_gene_expression_matrix.tsv", DATA_FILE)       # all 290 cells, for stage 1
    if REACTION_SCOPE not in ("all", "test"):
        SEL_FILE.write_text("\n".join(sorted(SCOPE_IDS)) + "\n")

    # ---- stage 1 ----------------------------------------------------------------------------------------------------------
    n_proc1 = max(1, min(os.cpu_count() or 1, 8))
    print(f"stage 1: penalties of all {len(expr_cols)} cells on {n_proc1} process(es) ... (about 0.5-2 minutes)")
    res = run_blocking(compass_args() + ["--only-penalties", "--num-processes", str(n_proc1)], "stage1")
    pen_file, pdir = TMP / "penalties.txt.gz", TMP / "penalties_tmp" / "penalties"
    if res["timeout"] or res["code"] != 0 or not pen_file.exists():
        explain_failure("[FAIL] stage 1 did not finish: " + ("timed out" if res["timeout"] else f"exit code {res['code']}"))
        return False
    pen = pd.read_csv(pen_file, sep="\t", index_col=0)
    problems = []
    if list(pen.columns) != expr_cols:                      problems.append("the penalty table lists other cells (or another order) than the expression file")
    if pen.shape[0] != len(STATE["cm"].reactions):          problems.append(f"{pen.shape[0]:,} penalty rows instead of {len(STATE['cm'].reactions):,} reactions")
    if len(list(pdir.glob("sample*/penalties.txt.gz"))) != len(expr_cols):
        problems.append("the number of per-cell penalty files is not the number of cells")
    if problems:
        print("[FAIL] stage 1 output is not what was expected:\n   " + "\n   ".join(problems)); return False
    print(f"stage 1 finished in {res['secs']:.0f} s: penalty table {pen.shape[0]:,} reactions x {pen.shape[1]} cells")

    # ---- the position trap: stage 2 reads 'sample{i}' for the i-th column of the data file it is given ----------------------
    # So the penalty file of the chosen cell with original position positions[j] must become 'sample{j}'.
    allp = pdir.parent / "penalties_all_cells"
    pdir.rename(allp)                                        # keep the 290 per-cell files untouched, as a record
    pdir.mkdir()
    for j, pos in enumerate(positions):
        (pdir / f"sample{j}").mkdir()
        shutil.copyfile(allp / f"sample{pos}" / "penalties.txt.gz", pdir / f"sample{j}" / "penalties.txt.gz")
    wrong = []                                               # verify BY NAME and by value against the stage-1 table
    for j, c in enumerate(cells):
        one = pd.read_csv(pdir / f"sample{j}" / "penalties.txt.gz", sep="\t", index_col=0)
        ok = (list(one.columns) == [c]) and one.index.equals(pen.index) and np.allclose(one.iloc[:, 0].to_numpy(float), pen[c].to_numpy(float), rtol=1e-12, atol=0)
        if not ok:
            wrong.append(f"sample{j} should hold {c} (column {positions[j]})")
    if wrong:
        print("[FAIL] the penalty files do not belong to the right cells - NOT starting:\n   " + "\n   ".join(wrong[:5])); return False
    moved = sum(1 for j, pos in enumerate(positions) if j != pos)
    print(f"penalties of the {len(cells)} chosen cells placed at positions 0..{len(cells) - 1} and verified by cell name and by value ({moved} of them had to move).")

    # ---- the data file shrinks to the chosen cells (same path, so Compass sees identical options) ---------------------------
    full = pd.read_csv(DATA_FILE, sep="\t", index_col=0)
    full[cells].to_csv(DATA_FILE, sep="\t")
    assert list(pd.read_csv(DATA_FILE, sep="\t", index_col=0, nrows=1).columns) == cells
    PREP_FILE.write_text(json.dumps({"cells": cells, "positions": positions, "scope": scope_tag, "seed": SEED,
                                     "stage1_seconds": round(res["secs"], 1), "prepared": time.strftime("%Y-%m-%d %H:%M:%S")}, indent=1))
    print(f"data file now holds the {len(cells)} chosen cells.  Prepared: {PREP_FILE}")
    return True

### C.2 Prepare: stage 1 and the penalty placement
Run this cell. It takes about a minute and prints what it did. If it ends with `[FAIL]`, the message says what to fix (the pre-flight in Part A must have passed first).

In [ ]:
if not preflight_ok:
    print("Pre-flight (A.3) has not passed - fix that first.")
else:
    prepared = prepare()

### C.3 Launch stage 2 in the background
Compass starts as its **own process** (`start_new_session`): closing the notebook or VS Code does **not** stop it. It stops if the Mac **shuts down or sleeps** — with `KEEP_AWAKE = True` the notebook runs `caffeinate` for you, which prevents *idle* sleep (a **closed lid** still sleeps unless the Mac is on power with an external display).

Before it starts, the cell checks: pre-flight passed · stage 1 prepared · the estimated time from Part B is within `MAX_HOURS` · Compass is not already running · the run is not already finished. **Re-running this cell after an interruption resumes the run**: cells that finished keep their results (their `success_token`), the others start again **[Repo, algorithm.py `singleSampleCompass`]**.

In [ ]:
def launch():
    if not preflight_ok:
        return print("Pre-flight (A.3) has not passed - fix that first.")
    if not PREP_FILE.exists():
        return print("Run the prepare cell (C.2) first.")
    if (RUN / "reactions.tsv").exists():
        return print(f"Compass has already finished this experiment ({RUN / 'reactions.tsv'}). Go to Part E.")
    st = read_run_state()
    if st and pid_alive(st["pid"]):
        return print(f"Compass is already running (process {st['pid']}). Look at Part D; nothing was started.")
    if STATE.get("est_hours", 0) > MAX_HOURS:
        return print(f"! The estimated time ({STATE['est_hours']:.1f} h) is longer than MAX_HOURS = {MAX_HOURS}. Nothing was started.\n"
                     "  Reduce N_PER_GROUP / REACTION_SCOPE (Part A.2), or raise MAX_HOURS on purpose.")
    prep = json.loads(PREP_FILE.read_text())
    n = len(prep["cells"])
    done0 = sum((TMP / f"sample{j}" / "success_token").exists() for j in range(n))
    n_proc = min(N_PROC_USED, n)
    LOG_DIR.mkdir(parents=True, exist_ok=True)
    tag = time.strftime("%Y%m%d_%H%M%S")
    with open(LOG_DIR / f"stage2_{tag}.out.txt", "w") as fo, open(LOG_DIR / f"stage2_{tag}.err.txt", "w") as fe:
        p = subprocess.Popen(STATE["cmd"] + compass_args() + ["--num-processes", str(n_proc)], cwd=RUN,
                             env={**STATE["env"], "PYTHONUNBUFFERED": "1"}, stdout=fo, stderr=fe, stdin=subprocess.DEVNULL,
                             start_new_session=True)
    awake = None
    if KEEP_AWAKE and shutil.which("caffeinate"):
        awake = subprocess.Popen(["caffeinate", "-i", "-w", str(p.pid)], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                                 stdin=subprocess.DEVNULL, start_new_session=True).pid
    STATE_FILE.write_text(json.dumps({
        "pid": p.pid, "caffeinate_pid": awake, "started_epoch": time.time(), "started": time.strftime("%Y-%m-%d %H:%M:%S"),
        "done_at_launch": done0, "n_cells": n, "n_proc": n_proc, "est_hours": STATE.get("est_hours"),
        "loop_total": len(evaluated),                                                              # size of Compass' reaction progress bar
        "w_reactions": float(np.clip(len(need_lp) / max(lp_per_cell, 1), 0.5, 0.97)),             # share of a cell's work in that bar
        "stdout": str(LOG_DIR / f"stage2_{tag}.out.txt"), "stderr": str(LOG_DIR / f"stage2_{tag}.err.txt")}, indent=1))
    print(f"LAUNCHED  Compass stage 2: {n} cells ({done0} already finished) on {n_proc} process(es), process id {p.pid}.")
    print(f"   estimated time {(STATE.get('est_hours') or float('nan')) * 60:.0f} min  [Inference]" + ("   |   Mac kept awake with caffeinate" if awake else ""))
    print("   Compass is a separate process: closing the notebook or VS Code does not stop it. Part D shows the progress; when it says FINISHED, run Parts E-G.")
    return True

launch()

## Part D · Progress
**Run the cell below whenever you like** — it only *reads* what Compass leaves in the experiment folder, so it is instant and never disturbs the run. It also works right after a kernel restart (run A.1, A.2 and A.2b first). With `WAIT_IN_NOTEBOOK = True` (A.2) and Compass running, this cell **stays busy and refreshes the report every minute** until Compass is finished; the square "interrupt" button stops the watching only — Compass itself keeps running.

How the numbers are obtained: a cell counts as **finished** when Compass has written its `success_token`; a cell that is still running is estimated from the progress bar Compass writes to `_tmp/sample<i>/err.log` (reaction `k` of `n`). The finishing time is extrapolated from the speed since the launch, so it is **[Inference]** — rough at first, better after a few cells have finished.

In [ ]:
import signal, datetime

def cell_fraction(j, loop_total, w_react):
    """Fraction (0-1) of cell j's work that is done, from its success_token or the last tqdm progress bar in its err.log."""
    if (TMP / f"sample{j}" / "success_token").exists():
        return 1.0
    f = TMP / f"sample{j}" / "err.log"
    if not f.exists():
        return 0.0
    with open(f, "rb") as fh:
        fh.seek(0, 2); size = fh.tell(); fh.seek(max(0, size - 20000))
        tail = fh.read().decode(errors="replace")
    hits = re.findall(r"(\d+)/(\d+) \[", tail)               # tqdm prints "  37%|###   | 370/1000 [01:02<01:50,  5.7it/s]"
    if not hits:
        return 0.0                                           # still building the model
    cur, tot = int(hits[-1][0]), int(hits[-1][1])
    return w_react * cur / tot if tot == loop_total else w_react + (1 - w_react) * cur / tot

def hms(seconds):
    s = int(max(0, seconds)); return f"{s // 3600}:{s % 3600 // 60:02d}:{s % 60:02d}"

def status():
    if not PREP_FILE.exists():
        return print("Nothing is prepared yet: run Part C (C.2).")
    n = len(json.loads(PREP_FILE.read_text())["cells"])
    st = read_run_state()
    finished = (RUN / "reactions.tsv").exists()
    if st is None:
        return print("Stage 2 has not been launched yet: run the launch cell (C.3).")
    alive = pid_alive(st["pid"])
    fr = [cell_fraction(j, st["loop_total"], st["w_reactions"]) for j in range(n)]
    done = [j for j in range(n) if fr[j] >= 1.0]
    running = [j for j in range(n) if 0 < fr[j] < 1.0]
    total, el = sum(fr), time.time() - st["started_epoch"]
    state = "FINISHED" if finished else ("RUNNING" if alive else "STOPPED before finishing")
    print(f"{state}   (process {st['pid']}, launched {st['started']}" + (f", {hms(el)} elapsed)" if alive else ")"))
    print(f"cells finished: {len(done)}/{n}   overall progress: {100 * total / n:.0f} %")
    if running:
        print("in progress   : " + ", ".join(f"cell {j} ({100 * fr[j]:.0f} %)" for j in running))
    waiting = [j for j in range(n) if fr[j] == 0.0]
    if waiting and alive:
        print(f"waiting/starting: {len(waiting)} cell(s)")
    if alive and (total - st["done_at_launch"]) >= 0.3:
        rem = el * (n - total) / (total - st["done_at_launch"])
        print(f"estimated finish: about {hms(rem)} from now (around {(datetime.datetime.now() + datetime.timedelta(seconds=rem)):%H:%M})   [Inference: speed so far]")
    elif alive:
        print(f"estimated finish: not yet known (the first cells are still being solved); the Part B estimate was {(st.get('est_hours') or float('nan')) * 60:.0f} min")
    if finished:
        print(f"\nCompass wrote {RUN / 'reactions.tsv'}.  -> Run Parts E to G.")
    elif not alive:
        if len(done) == n:
            print("\nAll cells are finished but the final table was not written. Run the launch cell (C.3) again: it only collects the results.")
        else:
            explain_failure(f"\nCompass is no longer running and {n - len(done)} cell(s) are unfinished.")
            print("   -> After fixing the cause, run the launch cell (C.3) again: finished cells are kept, the others restart.")

def watch(every_s=60):
    """Refresh status() every `every_s` seconds until Compass has stopped. Stop it with the square 'interrupt' button."""
    from IPython.display import clear_output
    try:
        while True:
            clear_output(wait=True); status()
            st = read_run_state()
            if st is None or not pid_alive(st["pid"]):
                break
            time.sleep(every_s)
    except KeyboardInterrupt:
        print("(watching stopped; Compass keeps running)")

def stop_run():
    """Stop Compass and all its worker processes (finished cells are kept; run the launch cell again to resume)."""
    st = read_run_state()
    if not st or not pid_alive(st["pid"]):
        return print("Compass is not running.")
    try:
        os.killpg(os.getpgid(st["pid"]), signal.SIGTERM)
        for _ in range(20):
            time.sleep(0.5)
            if not pid_alive(st["pid"]): break
        else:
            os.killpg(os.getpgid(st["pid"]), signal.SIGKILL)
    except ProcessLookupError:
        pass
    print("Compass stopped. Finished cells are kept; run the launch cell (C.3) to resume.")

_st = read_run_state()
if WAIT_IN_NOTEBOOK and _st and pid_alive(_st["pid"]):
    watch()                                  # refreshes the report every minute until Compass is done; the square "interrupt" button only stops the watching
else:
    status()
# stop_run()    # <- remove the '#' to stop Compass now

## Part E · Is the output right?
**Run this part when Part D says FINISHED.** It asks three questions, in this order, because each depends on the one before:

1. **Is the file complete and well-formed?** (all chosen cells, every evaluated reaction, no missing values, none negative.)
2. **Does every cell agree with the authors' `reactions.tsv`?** Compass is deterministic, so the same cell with the same settings must give (almost) the same number. We compare **cell by cell**, not on average, because the trap described in the introduction would show up as *one cell disagreeing completely*. Per cell: the **largest relative difference** `|a − b| / max(|a|, |b|)` over all reactions, and the number of entries that are zero in one file and not in the other. **PASS** ≤ `TOL_PASS`, **WARN** ≤ `TOL_WARN`, otherwise **FAIL** (same rule as Notebook 4's level 6; Gurobi's solver tolerances alone give about 1e-6).
3. **How long did it take?** Compass logs the CPU seconds of each cell. From them the cell below projects the cost of bigger runs **[Inference]**.

In [ ]:
def partE():
    f = RUN / "reactions.tsv"
    if not (PREP_FILE.exists() and f.exists()):
        return print("Compass has not written reactions.tsv yet. Run Part D; when it says FINISHED, run this cell again.")
    run_cells = json.loads(PREP_FILE.read_text())["cells"]
    if run_cells != cells:
        return print("This experiment folder was prepared for other cells than the ones Part B chose now. Re-run Part B with the same N_PER_GROUP / SEED.")
    OUT.mkdir(parents=True, exist_ok=True)
    CHECKS[:] = [c for c in CHECKS if c["level"] < 5]                          # running this cell again replaces its earlier rows
    ours, ref, st = pd.read_csv(f, sep="\t", index_col=0), STATE["ref"], read_run_state()
    record(5, "Compass finished stage 2", "PASS", f"{ours.shape[1]} cells; reactions.tsv written {time.strftime('%Y-%m-%d %H:%M', time.localtime(f.stat().st_mtime))}")

    # ---- E.1 well-formed ------------------------------------------------------------------------------------------------
    expected = [r.id for r in evaluated if not r.is_exchange]                  # every evaluated non-exchange reaction must have a score
    missing = sorted(set(expected) - set(ours.index))
    problems = []
    if list(ours.columns) != run_cells:  problems.append("the columns are not the cells that were run (or in another order)")
    if ours.isna().any().any():          problems.append(f"{int(ours.isna().sum().sum())} missing values")
    if (ours.values < 0).any():          problems.append("negative penalties")
    if missing:                          problems.append(f"{len(missing)} evaluated reactions have no score, e.g. {missing[:3]}")
    record(6, "output file is well-formed and complete", "FAIL" if problems else "PASS",
           "; ".join(problems) if problems else f"{ours.shape[0]:,} reactions x {ours.shape[1]} cells; all {len(expected):,} evaluated reactions present, no missing values, none negative")
    if problems:
        return
    STATE["ours"] = ours

    # ---- E.2 cell-by-cell comparison with the authors' file --------------------------------------------------------------------
    rows = ours.index.intersection(ref.index)
    rec, pooled_rel = [], []
    for j, c in enumerate(run_cells):
        a, b = ours.loc[rows, c].to_numpy(float), ref.loc[rows, c].to_numpy(float)
        zero_diff = int(((a == 0) != (b == 0)).sum())
        nz = (a != 0) & (b != 0)
        rel = np.abs(a[nz] - b[nz]) / np.maximum(np.abs(a[nz]), np.abs(b[nz]))
        worst, med = (float(rel.max()), float(np.median(rel))) if rel.size else (np.nan, np.nan)
        status = "FAIL" if (rel.size == 0 or zero_diff > 0) else ("PASS" if worst <= TOL_PASS else ("WARN" if worst <= TOL_WARN else "FAIL"))
        rec.append({"cell": c, "group": group[c], "position in the 290-column file": positions[j], "scores compared": int(nz.sum()),
                    "largest relative difference": worst, "median relative difference": med, "zero pattern differs": zero_diff, "status": status})
        pooled_rel.append((b[nz], a[nz], rel))
    cmp_tab = pd.DataFrame(rec)
    display(cmp_tab)
    order = {"PASS": 0, "WARN": 1, "FAIL": 2}
    overall = max(cmp_tab.status, key=order.get)
    bad_cells = cmp_tab.loc[cmp_tab.status != "PASS", "cell"].tolist()
    record(6, "every cell agrees with the authors' reactions.tsv", overall,
           f"{int((cmp_tab.status == 'PASS').sum())}/{len(cmp_tab)} cells PASS; largest relative difference of all {cmp_tab['largest relative difference'].max():.1e} "
           f"(TOL_PASS {TOL_PASS:g}, TOL_WARN {TOL_WARN:g})" + (f"; not PASS: {', '.join(bad_cells[:5])}" if bad_cells else ""))
    if overall == "FAIL":
        print("   A FAIL in one or a few cells with large differences points to the wrong penalties for those cells (the position trap) - look at that cell's\n"
              f"   log in {TMP}/sample<j>/compass.log (j = its row in the table). FAIL in all cells points to a different Compass/Recon2 version or media.\n"
              "   Parts F-G below still run so that you can look at the results, but do not trust them until this is understood.")
    STATE["cmp"], STATE["e_status"] = cmp_tab, overall

    b_all = np.concatenate([x[0] for x in pooled_rel]); a_all = np.concatenate([x[1] for x in pooled_rel]); rel_all = np.concatenate([x[2] for x in pooled_rel])
    pick = np.random.default_rng(0).choice(len(b_all), min(60000, len(b_all)), replace=False) if len(b_all) else []
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    ax[0].loglog(b_all[pick], a_all[pick], ".", ms=3, color="#348C73", alpha=0.5)
    lim = [min(b_all.min(), a_all.min()), max(b_all.max(), a_all.max())]
    ax[0].plot(lim, lim, "k--", lw=1); ax[0].set_xlabel("authors' Compass penalty"); ax[0].set_ylabel("penalty from YOUR run")
    ax[0].set_title(f"A. one dot per (reaction, cell) - {len(b_all):,} scores" + (", 60,000 shown" if len(b_all) > 60000 else ""), fontsize=10)
    ax[1].hist(np.log10(rel_all + 1e-17), bins=40, color="#E92E87")
    ax[1].axvline(np.log10(TOL_PASS), color="k", ls="--"); ax[1].text(np.log10(TOL_PASS) - 0.15, ax[1].get_ylim()[1] * 0.9, "PASS limit", ha="right")
    ax[1].set_xlabel("log10 of the relative difference  (-17 = identical)"); ax[1].set_ylabel("number of scores"); ax[1].set_title("B. how big are the differences?", fontsize=10)
    plt.tight_layout(); fig.savefig(OUT / "E_agreement_with_authors.png", dpi=150, bbox_inches="tight"); plt.show()

    # ---- E.3 timing and projection ----------------------------------------------------------------------------------------------
    tim = []
    for j, c in enumerate(run_cells):
        log = TMP / f"sample{j}" / "compass.log"
        txt = log.read_text(errors="replace") if log.exists() else ""
        m1, m2 = re.search(r"Compass Reaction Time:\s*([\d.]+)", txt), re.search(r"Compass Exchange Time:\s*([\d.]+)", txt)
        tim.append({"cell": c, "LPs solved": int((ours.loc[expected, c] != 0).sum()),
                    "reaction loop (CPU s)": float(m1.group(1)) if m1 else np.nan, "exchange part (CPU s)": float(m2.group(1)) if m2 else np.nan})
    tim = pd.DataFrame(tim)
    STATE["timing"] = tim
    if tim["reaction loop (CPU s)"].notna().all() and tim["LPs solved"].sum() > 0:
        spl = tim["reaction loop (CPU s)"].sum() / tim["LPs solved"].sum()
        per_cell = (tim["reaction loop (CPU s)"] + tim["exchange part (CPU s)"]).mean()
        wall_meas = (f.stat().st_mtime - st["started_epoch"]) / 3600 if st else np.nan
        print(f"\nmeasured: {tim['LPs solved'].sum():,} LPs in {tim['reaction loop (CPU s)'].sum():,.0f} CPU-seconds  ->  {spl:.2f} s per LP;  "
              f"{per_cell / 60:.1f} CPU-minutes per cell (reaction loop + exchange part)")
        if st:
            note = "" if st["done_at_launch"] == 0 else "  (the run was resumed, so this under-states the real time)"
            print(f"wall-clock from launch to the last cell: {wall_meas * 60:.0f} min on {st['n_proc']} processes;  Part B predicted {(st.get('est_hours') or float('nan')) * 60:.0f} min{note}")
        full_lp = n_lp_full + n_exch_lp
        scen = [("this run", len(run_cells), lp_per_cell, per_cell / 3600 * len(run_cells)),
                ("same reactions, all 290 cells", 290, lp_per_cell, None),
                ("ALL reactions, the same cells", len(run_cells), full_lp, None),
                ("ALL reactions, all 290 cells (the paper's full run)", 290, full_lp, None)]
        proj = pd.DataFrame([{"scenario": s, "cells": nc, "LPs per cell": int(lp), "CPU-hours": (cpu if cpu is not None else nc * lp * spl / 3600)} for s, nc, lp, cpu in scen])
        proj[f"wall-clock hours on {N_PROC_USED} processes"] = proj["CPU-hours"] / N_PROC_USED
        proj["wall-clock hours on 16 processes"] = proj["CPU-hours"] / 16
        display(proj.round(2))
        print("[Inference] exchange reactions are counted as 4 LPs per metabolite with the same seconds per LP; only the scenarios 'ALL reactions' include them.\n"
              "The repo documentation says about 30 minutes of CPU per complete cell [Repo, docs/quickstart.rst].")
        STATE["projection"] = proj
    else:
        print("\nThe per-cell logs have no timing lines; no projection.")

partE()

## Part F · The biology: Th17p vs Th17n on **your** Compass output
Now the paper's question, answered from the 20 cells you ran. The statistics are exactly those of Notebook 1, §6 (functions in B.0): score = −log(1 + penalty) **[Paper, Alg. 3]**; for each reaction, **Cohen's d** (size of the Th17p − Th17n difference; positive = higher potential activity in Th17p), a **Wilcoxon rank-sum test**, and **Benjamini–Hochberg** adjusted p-values; "differentially active" = adjusted p < `ALPHA` **[Paper, STAR Methods]**.

The same reactions are analysed in **three** ways, so every difference has one cause:

| name | scores from | cells | answers |
|---|---|---|---|
| **yours** | your Compass run | your 20 | what you can learn from this experiment |
| **authors, same 20** | the authors' `reactions.tsv` | your 20 | must equal "yours" if Compass reproduced exactly; any difference is the run, not the sampling |
| **authors, all 290** | the authors' `reactions.tsv` | all 290 | the "truth" the paper reports; the gap between this and "yours" is the price of using 20 cells instead of 290 |

**How the comparison is made.** Only reactions that are *kept* in the paper's analysis (score range over the 290 cells ≥ `MIN_RANGE`) and that you evaluated are used — exactly the set of Part B.3, so the simulation there and your result are comparable. We then ask (i) do the effect sizes of your run **rank the reactions the same way** as the 290-cell result (Spearman correlation), and where does your correlation fall among the `N_DRAWS` random subsets of Part B.3? (ii) do the **signs** agree, and how does that depend on the size of the effect? (iii) for the 26 reactions the authors label in their Fig. 2C, where does your estimate fall, with a **bootstrap 95 % interval** (resample the cells of each group with replacement `N_BOOT` times — a way to see how much the estimate would move with different cells)?

In [ ]:
def bootstrap_d(A_p, A_n, n_boot, seed=1):
    """Cohen's d of every row for n_boot bootstrap resamples of the columns of each group -> array (rows, n_boot)."""
    r = np.random.default_rng(seed)
    P = A_p[:, r.integers(0, A_p.shape[1], (n_boot, A_p.shape[1]))]                # (rows, n_boot, n_p)
    N = A_n[:, r.integers(0, A_n.shape[1], (n_boot, A_n.shape[1]))]
    nA, nB = P.shape[2], N.shape[2]
    pooled = np.sqrt(((nA - 1) * P.var(axis=2, ddof=1) + (nB - 1) * N.var(axis=2, ddof=1)) / (nA + nB - 2))
    with np.errstate(divide="ignore", invalid="ignore"):
        d = (P.mean(axis=2) - N.mean(axis=2)) / pooled
    d[~np.isfinite(d)] = np.nan
    return d

def partF():
    if "ours" not in STATE:
        return print("Run Part E first (it reads and checks your output).")
    if STATE.get("e_status") == "FAIL":
        print("! Part E reported FAIL: the numbers below are shown for inspection only.\n")
    ours = STATE["ours"]
    rows_F = [r for r in rows_B if r in ours.index]                      # kept by the paper's filter AND evaluated by you
    all_p, all_n = list(group.index[group == "Th17p"]), list(group.index[group == "Th17n"])
    S_you = -np.log(ours.loc[rows_F] + 1.0)
    res_you  = differential_activity(S_you, cells_p_sub, cells_n_sub)
    res_same = differential_activity(S_auth.loc[rows_F], cells_p_sub, cells_n_sub)
    res_all  = differential_activity(S_auth.loc[rows_F], all_p, all_n)
    d_you, d_same, d_all = res_you.cohens_d, res_same.cohens_d, res_all.cohens_d
    ok = np.isfinite(d_you) & np.isfinite(d_same) & np.isfinite(d_all)
    rho      = d_you[ok].corr(d_all[ok], method="spearman")
    rho_same = d_you[ok].corr(d_same[ok], method="spearman")
    base = baseline.loc[baseline.n_per_group == N_PER_GROUP, "rho"].dropna()
    pct = float((base < rho).mean())

    sub = rxn_md.subsystem.reindex([base_id(r) for r in rows_F]).to_numpy()
    panel_of = {s: p for p, v in PANELS.items() for s in v}
    panel = pd.Series([panel_of.get(s, "other") for s in sub], index=rows_F)
    print(f"{len(rows_F):,} reactions analysed ({int(ok.sum()):,} with a finite Cohen's d in all three versions); {len(cells_p_sub)} Th17p + {len(cells_n_sub)} Th17n cells\n")
    print(f"Spearman correlation of Cohen's d, yours vs authors-same-20 : {rho_same:.4f}   (should be ~1: same cells, same numbers)")
    print(f"Spearman correlation of Cohen's d, yours vs all 290 cells   : {rho:.3f}   -> higher than {pct:.0%} of {len(base)} random {N_PER_GROUP}+{N_PER_GROUP} subsets of the authors' data "
          f"(Part B.3: median {base.median():.2f}, 10th-90th percentile {base.quantile(.1):.2f}-{base.quantile(.9):.2f})")
    print("   -> a value far BELOW the grey boxes of Part B.3 would point to a problem beyond bad luck in the choice of cells.\n")

    # ---- signs and effect size --------------------------------------------------------------------------------------------------
    edges, names = [0, 0.2, 0.5, 0.8, np.inf], ["< 0.2", "0.2 - 0.5", "0.5 - 0.8", ">= 0.8"]
    cat = pd.cut(d_all.abs(), edges, labels=names, right=False)
    sign_tab = pd.DataFrame({"reactions": d_all.groupby(cat, observed=False).size(),
                             "same sign as in the 290 cells": (np.sign(d_you) == np.sign(d_all)).groupby(cat, observed=False).mean(),
                             "median |d yours - d 290|": (d_you - d_all).abs().groupby(cat, observed=False).median()})
    sign_tab.index.name = "|d| in the 290 cells"
    print("Do the signs agree?  (bins of the size of the effect in all 290 cells; 0.2 / 0.5 / 0.8 are the conventional 'small / medium / large' labels of Cohen (1988), "
          "Statistical Power Analysis for the Behavioral Sciences, 2nd ed. - a convention, not a result of the paper)")
    display(sign_tab.round(3))

    # ---- significance -------------------------------------------------------------------------------------------------------
    s_you, s_same, s_all = res_you.adj_p < ALPHA, res_same.adj_p < ALPHA, res_all.adj_p < ALPHA
    sig = pd.DataFrame({"reactions": [int(s_all.sum()), int(s_same.sum()), int(s_you.sum()), int((s_you & s_all).sum()), int((s_you & s_same).sum())]},
                       index=[f"significant (adj. p < {ALPHA:g}) with all 290 cells", "significant with the authors' scores of your 20 cells", "significant with YOUR run (20 cells)",
                              "significant with your run AND with 290 cells", "significant with your run AND with the authors' same 20"])
    display(sig)
    if s_all.sum():
        print(f"Of the {int(s_all.sum())} reactions significant with 290 cells, your {len(cells)} cells recover {(s_you & s_all).sum() / s_all.sum():.0%} (the 'power' of the small experiment).")
    if s_you.sum():
        print(f"Of the {int(s_you.sum())} reactions significant in your run, {(s_you & s_all).sum() / s_you.sum():.0%} are also significant with 290 cells.")

    # ---- pathway level ----------------------------------------------------------------------------------------------------------
    pt = []
    for p in list(PANELS) + ["other"]:
        m = (panel == p) & ok
        if m.sum():
            pt.append({"panel": p, "reactions": int(m.sum()), "median d (yours)": d_you[m].median(), "median d (290 cells)": d_all[m].median(),
                       "same sign": float((np.sign(d_you[m]) == np.sign(d_all[m])).mean())})
    pan = pd.DataFrame(pt).set_index("panel")
    print("\nPathway level (positive d = higher potential activity in Th17p):")
    display(pan.round(3))

    # ---- figure F1 ----------------------------------------------------------------------------------------------------------------
    fig, ax = plt.subplots(1, 3, figsize=(16, 4.6), gridspec_kw={"width_ratios": [1.15, 1, 1]})
    for p in ["other"] + list(PANELS):
        m = (panel == p) & ok
        ax[0].scatter(d_all[m], d_you[m], s=10 if p == "other" else 16, color=PANEL_COLOUR[p], alpha=0.6 if p == "other" else 0.85, label=f"{p} ({int(m.sum())})", zorder=2 if p != "other" else 1)
    lim = float(np.nanmax(np.abs(np.r_[d_all[ok], d_you[ok]]))) * 1.05
    ax[0].plot([-lim, lim], [-lim, lim], "k--", lw=1); ax[0].axhline(0, color="grey", lw=.5); ax[0].axvline(0, color="grey", lw=.5)
    ax[0].set_xlabel("Cohen's d, all 290 cells (authors)"); ax[0].set_ylabel(f"Cohen's d, YOUR {len(cells)} cells")
    ax[0].set_title(f"A. one dot per reaction;  Spearman rho = {rho:.2f}", fontsize=10); ax[0].legend(frameon=False, fontsize=7, loc="upper left")
    plot_baseline(ax[1], mark=rho); ax[1].set_title(f"B. your rho against {len(base)} random subsets", fontsize=10)
    x = np.arange(len(pan)); w = 0.38
    ax[2].bar(x - w / 2, pan["median d (yours)"], w, color="#E92E87", label="yours"); ax[2].bar(x + w / 2, pan["median d (290 cells)"], w, color="grey", label="all 290 cells")
    ax[2].axhline(0, color="k", lw=.6); ax[2].set_xticks(x); ax[2].set_xticklabels([t.replace(" ", "\n", 1) for t in pan.index], fontsize=8)
    ax[2].set_ylabel("median Cohen's d (+ = higher in Th17p)"); ax[2].legend(frameon=False); ax[2].set_title("C. pathway medians", fontsize=10)
    plt.tight_layout(); fig.savefig(OUT / "F1_yours_vs_290_cells.png", dpi=150, bbox_inches="tight"); plt.show()

    # ---- the 26 reactions the authors label in Fig. 2C ------------------------------------------------------------------------------------
    lab = [r for r in LABELED if r in rows_F]
    if len(lab) < len(LABELED):
        print(f"\n{len(LABELED) - len(lab)} of the 26 labelled reactions are not in your analysis (not in the chosen scope or filtered out): "
              f"{[r for r in LABELED if r not in lab]}")
    lab_tab = None
    if lab:
        A_p, A_n = S_you.loc[lab, cells_p_sub].to_numpy(float), S_you.loc[lab, cells_n_sub].to_numpy(float)
        boot = bootstrap_d(A_p, A_n, N_BOOT)
        lo, hi = np.nanpercentile(boot, 2.5, axis=1), np.nanpercentile(boot, 97.5, axis=1)
        lab_tab = pd.DataFrame({"enzyme": [LABELED[r] for r in lab], "d yours": d_you[lab].to_numpy(), "95% interval low": lo, "95% interval high": hi,
                                "d 290 cells": d_all[lab].to_numpy(), "adj. p yours": res_you.loc[lab, "adj_p"].to_numpy(), "adj. p 290 cells": res_all.loc[lab, "adj_p"].to_numpy()}, index=lab)
        lab_tab["290-cell value inside your interval"] = (lab_tab["d 290 cells"] >= lab_tab["95% interval low"]) & (lab_tab["d 290 cells"] <= lab_tab["95% interval high"])
        lab_tab["same sign"] = np.sign(lab_tab["d yours"]) == np.sign(lab_tab["d 290 cells"])
        lab_tab = lab_tab.sort_values("d 290 cells")
        display(lab_tab.round(3))
        print(f"{int(lab_tab['same sign'].sum())}/{len(lab_tab)} labelled reactions have the same sign as with 290 cells; the 290-cell value lies inside your {N_BOOT}-resample 95% interval for "
              f"{int(lab_tab['290-cell value inside your interval'].sum())}/{len(lab_tab)}.  (An interval is wide when 10 cells per group leave the effect uncertain.)")
        fig, ax = plt.subplots(figsize=(8, 0.34 * len(lab_tab) + 1.6))
        y = np.arange(len(lab_tab))
        ax.hlines(y, lab_tab["95% interval low"], lab_tab["95% interval high"], color=PAL["Th17p"], lw=2, alpha=.6)
        ax.scatter(lab_tab["d yours"], y, color=PAL["Th17p"], zorder=3, label=f"yours ({len(cells)} cells), 95% bootstrap interval")
        ax.scatter(lab_tab["d 290 cells"], y, marker="D", color="black", zorder=4, label="all 290 cells (authors)")
        ax.axvline(0, color="grey", lw=.6); ax.set_yticks(y); ax.set_yticklabels([f"{LABELED[r]}  ({r})" for r in lab_tab.index], fontsize=8)
        ax.set_xlabel("Cohen's d  (+ = higher potential activity in Th17p)"); ax.legend(frameon=False, fontsize=8, loc="lower right"); ax.set_title("The reactions labelled in the paper's Fig. 2C", fontsize=10)
        plt.tight_layout(); fig.savefig(OUT / "F2_labelled_reactions.png", dpi=150, bbox_inches="tight"); plt.show()

    # ---- the table with everything, one row per reaction ----------------------------------------------------------------------------------------
    diff = pd.DataFrame({"reaction_name": rxn_md.reaction_name.reindex([base_id(r) for r in rows_F]).to_numpy(), "subsystem": sub, "panel": panel.to_numpy(),
                         "d_yours": d_you, "p_yours": res_you.p, "adj_p_yours": res_you.adj_p,
                         "d_authors_same_cells": d_same, "adj_p_authors_same_cells": res_same.adj_p,
                         "d_290_cells": d_all, "adj_p_290_cells": res_all.adj_p}, index=rows_F)
    diff.index.name = "reaction"
    STATE["F"] = dict(rho=rho, rho_same=rho_same, pct=pct, base=base, sign=sign_tab, sig=sig, pan=pan, lab=lab_tab, diff=diff, n_rows=len(rows_F),
                      s_all=int(s_all.sum()), s_you=int(s_you.sum()), s_both=int((s_you & s_all).sum()))

partF()

**How to read Part F.**
* **Figure A** — every dot is one reaction. On the dashed diagonal your 20 cells and the 290 cells would agree exactly. Scatter around it is the sampling noise of using 20 cells; it is larger for reactions with small effects (near the centre) and the cloud tilting away from the diagonal would mean your subset *exaggerates or shrinks* effects. Colours are the four pathways of the paper's Fig. 2C.
* **Figure B** — the grey boxes are the correlations you could have got from **other random choices of 10 + 10 cells** (Part B.3, authors' data); the black star is yours. A star inside the boxes means your result is as good as a typical subset of that size can be; far below means unlucky cells (or a problem — check Part E).
* **Figure C** — the median effect per pathway. With only 20 cells this is the most reliable level of the analysis, because a median over hundreds of reactions averages out the noise of single ones.
* **Labelled reactions** — a reaction whose interval contains 0 cannot be called higher or lower in Th17p from your 20 cells alone.
* **Few or no significant reactions is normal with 10 + 10 cells.** The smallest possible two-sided rank-sum p-value for 10 vs 10 cells is 2 / C(20,10) = 2 / 184,756 ≈ 1e-5, and Benjamini–Hochberg multiplies it by (number of tests) / (rank) — so with about a thousand tests only very clear, consistent differences pass adj. p < 0.1. Rely on the effect sizes (Figures A–C) and raise `N_PER_GROUP` if you need significance calls. **[Inference — arithmetic of the test]**
* **What would make you doubt a result:** "yours vs authors-same-20" far from 1 (then the run differs from the authors', see Part E), or a correlation with the 290 cells below the grey boxes.

## Part G · Save the tables and figures, and the verdict
Tables are written in **Excel's European format** (`;` between columns, `,` as the decimal mark, UTF-8 with a byte-order mark so that Excel shows special characters correctly); figures as PNG. Everything goes to the `results/` folder of this experiment.

In [ ]:
def partG():
    if "F" not in STATE:
        return print("Run Parts E and F first.")
    OUT.mkdir(parents=True, exist_ok=True)
    F = STATE["F"]
    def save_eu(df, name, index=True):
        df.to_csv(OUT / name, sep=";", decimal=",", index=index, encoding="utf-8-sig")
    save_eu(STATE["ours"], "01_your_compass_penalties.csv")
    save_eu(STATE["cmp"], "02_agreement_with_authors_per_cell.csv", index=False)
    save_eu(F["diff"], "03_differential_activity_yours_vs_authors.csv")
    if F["lab"] is not None:
        save_eu(F["lab"], "04_labelled_reactions.csv")
    save_eu(F["pan"], "05_pathway_summary.csv")
    save_eu(F["sign"], "06_sign_agreement_by_effect_size.csv")
    save_eu(F["sig"], "07_significance_summary.csv")
    save_eu(STATE["timing"], "08_timing_per_cell.csv", index=False)
    if "projection" in STATE:
        save_eu(STATE["projection"].round(3), "09_projection_of_bigger_runs.csv", index=False)
    save_eu(baseline.groupby("n_per_group")["rho"].describe(percentiles=[.1, .5, .9]), "10_noise_baseline_spearman_by_subset_size.csv")
    (OUT / "00_settings.json").write_text(json.dumps({
        "N_PER_GROUP": N_PER_GROUP, "SEED": SEED, "REACTION_SCOPE": str(REACTION_SCOPE), "cells": cells, "positions_in_290_file": positions,
        "compass_options": " ".join(compass_args()), "ALPHA": ALPHA, "MIN_RANGE": MIN_RANGE, "TOL_PASS": TOL_PASS, "TOL_WARN": TOL_WARN,
        "spearman_vs_290_cells": F["rho"], "spearman_vs_authors_same_cells": F["rho_same"]}, indent=1))
    print(f"Saved in {OUT}:")
    for p in sorted(OUT.iterdir()):
        print(f"   {p.name:<58}{p.stat().st_size / 1e3:9.1f} kB")

    # ---- scoreboard and verdict ----------------------------------------------------------------------------------------------------
    import textwrap
    print()
    for r in CHECKS:
        print(f"{ICON[r['status']]}  L{r['level']}  {r['check']}")
        if r["detail"]:
            print(textwrap.fill(r["detail"], width=130, initial_indent="           ", subsequent_indent="           "))
    n = pd.Series([r["status"] for r in CHECKS]).value_counts().to_dict()
    print(f"\nPASS {n.get('PASS', 0)}   WARN {n.get('WARN', 0)}   FAIL {n.get('FAIL', 0)}   SKIP {n.get('SKIP', 0)}")
    if n.get("FAIL"):
        print("\nVERDICT: NOT RELIABLE - at least one check failed (first [FAIL] above). Do not interpret Part F before this is understood.")
    else:
        word = "reproduces the authors' numbers cell by cell" if not n.get("WARN") else "agrees with the authors' numbers, with warnings (read the WARN rows)"
        print(f"\nVERDICT: your Compass run {word}.")
        print(f"   Biology with {len(cells)} cells: Spearman rho = {F['rho']:.2f} with the 290-cell effect sizes (higher than {F['pct']:.0%} of random subsets of this size); "
              f"{F['s_both']}/{F['s_all']} of the reactions significant with 290 cells are found again.")

partG()

## What to do next
* **Looks right (Part E all PASS) and you want more cells:** change `N_PER_GROUP` in A.2 (for example 30 → 60 cells). The new experiment gets its own folder; Part B tells you the estimated time first. Part B.3's boxes show how much each doubling of the cells is expected to buy.
* **Want all reactions:** set `REACTION_SCOPE = "all"` (about 5× the cost per cell, plus the exchange reactions). Use the projection table of Part E to decide.
* **The paper's full run (290 cells, all reactions):** use the command in `README_SETUP.md` §2.6 on a machine with many cores; the projection table is your estimate for this computer.
* **Run stopped or Mac slept:** run Part D to see where it stopped, then the launch cell (C.3) again — finished cells are kept.
* **Start from scratch:** delete the experiment folder printed at the top of A.2.
* **Files in the experiment folder:** `input/` (data and reaction list given to Compass) · `_tmp/` (Compass' working files: `penalties.txt.gz` = stage 1, `sample<j>/` = per-cell logs and results) · `logs/` (console output of each stage) · `reactions.tsv` (**Compass' result**) · `results/` (tables and figures from Parts E–G) · `prepared.json`, `run_state.json` (bookkeeping).